# Thermal Data Collection

Derive the Mars coordinates of every image in `data/final_data/image_dataset.json`
and turn them into a THEMIS retrieval plan, saved under `data/thermal_data/`.

The pipeline this feeds:

1. **This notebook** — image -> centre latitude/longitude -> deduplicated query sites.
2. **`themis_measurements.py`** (Phase 1) — site -> THEMIS observation IDs covering it,
   with acquisition time and Mars local solar time.
3. **`themis_id_to_thermal_value.py`** (Phase 2) — observation ID -> decoded
   brightness-temperature matrix (`.npy`).

The model consumes a small window at **native THEMIS resolution (~100 m/pixel)**,
not a raster upsampled to the HiRISE grid, so what we need per site is a
`32 x 32` THEMIS patch per timestep rather than a full-resolution image.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import json
import subprocess
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from themis_coordinates import center_latlon, ground_extent_m, THEMIS_GSD_M

DATA_PATH = "data"
DATASET_JSON = os.path.join(DATA_PATH, "final_data", "image_dataset.json")

THERMAL_DIR = os.path.join(DATA_PATH, "thermal_data")
THEMIS_DIR = os.path.join(THERMAL_DIR, "themis_data")      # real THEMIS products
GENERATED_DIR = os.path.join(THERMAL_DIR, "generated_data")  # synthetic thermal

os.makedirs(THEMIS_DIR, exist_ok=True)

print("Thermal root :", THERMAL_DIR)
print("THEMIS output:", THEMIS_DIR)

C:\Users\Utente\Desktop\lavatube_paper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Thermal root : data\thermal_data
THEMIS output: data\thermal_data\themis_data


## 1. Load the training dataset

These are the same rows the model trains on, so every image gets a coordinate.

In [2]:
image_data = pd.read_json(DATASET_JSON)

class_names = {
    0: "Vertical Pit/Skylight",
    1: "Shallow Pit Chain",
    2: "Sloped Pit",
    3: "Plain Terrain",
}

print(f"{len(image_data)} images")
print(image_data["category_id"].map(class_names).value_counts().to_string())

2374 images
category_id
Shallow Pit Chain        657
Vertical Pit/Skylight    625
Sloped Pit               564
Plain Terrain            528


## 2. Coordinates

`image_dataset.json` already carries `lat`, `lon_east` and each crop's ground
extent, all computed once when the two source datasets were merged in
`data_overview.ipynb`. Nothing here needs to touch a raster.

Those coordinates come from `themis_coordinates.center_latlon`, which reads each
crop's own georeferencing and repairs a malformed CRS present in the
plain-terrain products: the standard parallel was written into
`latitude_of_origin` with `standard_parallel_1` left at 0, a known planetary-PDS
quirk. Read literally, that offsets latitude by exactly the standard parallel
(up to 45 deg here, ~2600 km) and mis-scales longitude. Section 3 validates the
repair against independently derived values.

If an older `image_dataset.json` without those columns is loaded, the cell falls
back to reading the rasters (slow: a couple of minutes for ~2400 files).

In [3]:
PRECOMPUTED = ["lat", "lon_east", "ground_width_m", "ground_height_m"]
has_coords = set(PRECOMPUTED).issubset(image_data.columns)

if has_coords:
    print("Using precomputed coordinates and extents from image_dataset.json")
    coordinates = image_data[
        ["image_name", "img_path", "category_id", *PRECOMPUTED]
    ].copy()
else:
    print("Columns missing -- recomputing from the rasters "
          "(re-run data_overview.ipynb to avoid this)")
    records = []
    for row in tqdm(
        image_data.itertuples(index=False),
        total=len(image_data),
        desc="Reading georeferencing",
    ):
        path = os.path.join(row.img_path, row.image_name)
        lat, lon = center_latlon(path)
        width_m, height_m = ground_extent_m(path)
        records.append({
            "image_name": row.image_name,
            "img_path": row.img_path,
            "category_id": row.category_id,
            "lat": lat,
            "lon_east": lon,
            "ground_width_m": width_m,
            "ground_height_m": height_m,
        })
    coordinates = pd.DataFrame(records)

coordinates["themis_px_across"] = coordinates["ground_width_m"] / THEMIS_GSD_M
coordinates.head()

Using precomputed coordinates and extents from image_dataset.json


,image_name,img_path,category_id,lat,lon_east,ground_width_m,ground_height_m,themis_px_across
0,ESP_011386_2065_RED_resized_0.5_lbl_0.tiff,data/DeepLandforms_dataset,0,26.148686,259.343078,864.5000,864.0,8.645000
1,ESP_011386_2065_RED_resized_0.5_lbl_1.tiff,data/DeepLandforms_dataset,1,26.124130,259.347565,1058.0000,1058.0,10.580000
2,ESP_011386_2065_RED_resized_1_lbl_0.tiff,data/DeepLandforms_dataset,0,26.148829,259.342953,1212.0000,1212.0,12.120000
3,ESP_011386_2065_RED_resized_1_lbl_1.tiff,data/DeepLandforms_dataset,1,26.124257,259.347462,1540.0000,1540.0,15.400000
4,ESP_011386_2065_RED_resized_2_lbl_0.tiff,data/DeepLandforms_dataset,0,26.148829,259.342960,2016.2625,2016.0,20.162625


## 3. Validate against the label-derived coordinates

The plain-terrain crops ship with `lat_center` / `lon_center` computed
independently from the source PDS label bounds, which gives us ground truth for
that half of the dataset.

Expect ~0 difference for the Equirectangular products. The Polar Stereographic
ones differ by a few tenths of a degree because the CSV interpolates linearly
between label corners, which degrades near the poles -- there the value derived
here is the more accurate of the two.

In [4]:
plain_truth = pd.read_csv(
    os.path.join(DATA_PATH, "plain_terrain_dataset", "plain_terrain_annotations.csv")
)

check = coordinates.merge(
    plain_truth[["image_name", "file_name", "lat_center", "lon_center"]],
    on="image_name",
    how="inner",
)

check["d_lat"] = (check["lat"] - check["lat_center"]).abs()
check["d_lon"] = ((check["lon_east"] - check["lon_center"] % 360 + 180) % 360 - 180).abs()

print(f"Cross-checked {len(check)} plain-terrain crops\n")
print(
    check.groupby("file_name")[["d_lat", "d_lon"]]
    .max()
    .round(4)
    .to_string()
)

Cross-checked 528 plain-terrain crops

                      d_lat   d_lon
file_name                          
ESP_011287_2165_RED  0.0000  0.0000
ESP_011293_1710_RED  0.0000  0.0000
ESP_011325_1845_RED  0.0000  0.0000
ESP_011335_1005_RED  0.2511  0.3818
ESP_011999_1305_RED  0.0000  0.0000
ESP_012016_1800_RED  0.0000  0.0000
ESP_043599_1650_RED  0.0000  0.0000
ESP_087433_2545_RED  0.0754  0.3644


## 4. How much thermal signal is actually there?

At ~100 m/pixel a HiRISE crop covers only a handful of THEMIS samples. This is
why the thermal stream gets its own small encoder instead of sharing the HiRISE
backbone, and why we extract a fixed window around each site rather than
matching the crop's exact footprint: a fixed `32 x 32` window (~3.2 km) is a
consistent tensor shape and includes surrounding terrain, which is what makes
the pit-versus-background temperature contrast measurable.

In [5]:
spans = coordinates["themis_px_across"]

print("THEMIS pixels across a crop:")
print(f"  min    {spans.min():6.1f}")
print(f"  median {spans.median():6.1f}")
print(f"  max    {spans.max():6.1f}")
print()
print(f"Crops smaller than one THEMIS pixel: {(spans < 1).sum()}")
print(f"Fixed extraction window: 32 x 32 THEMIS px "
      f"= {32 * THEMIS_GSD_M / 1000:.1f} x {32 * THEMIS_GSD_M / 1000:.1f} km")

THEMIS pixels across a crop:
  min       0.4
  median   16.0
  max      80.1

Crops smaller than one THEMIS pixel: 56
Fixed extraction window: 32 x 32 THEMIS px = 3.2 x 3.2 km


## 5. Deduplicate into query sites

Crops closer together than a THEMIS pixel resolve to the same thermal data, and
many crops are multi-resolution replicas of the same landform. Grouping them
onto a coarse grid collapses the number of network queries substantially --
Phase 1 and Phase 2 both hit remote archives, so this matters.

The grid step below is half the extraction window, so any two crops sharing a
site also share almost all of their thermal window.

In [6]:
GRID_DEG = 0.05  # ~3 km at the equator, about half the 32-px window

coordinates["site_lat"] = (coordinates["lat"] / GRID_DEG).round() * GRID_DEG
coordinates["site_lon"] = (coordinates["lon_east"] / GRID_DEG).round() * GRID_DEG

query_sites = (
    coordinates
    .groupby(["site_lat", "site_lon"], as_index=False)
    .agg(
        n_images=("image_name", "size"),
        categories=("category_id", lambda s: sorted(set(s))),
    )
    .sort_values(["site_lat", "site_lon"])
    .reset_index(drop=True)
)

query_sites["site_id"] = [f"site_{i:04d}" for i in range(len(query_sites))]

print(f"{len(coordinates)} images -> {len(query_sites)} unique query sites")
query_sites.head()

2374 images -> 357 unique query sites


,site_lat,site_lon,n_images,categories,site_id
0,-79.75,234.95,1,[3],site_0000
1,-79.75,235.10,1,[3],site_0001
2,-79.75,235.15,1,[3],site_0002
3,-79.70,234.95,1,[3],site_0003
4,-79.70,235.20,1,[3],site_0004


## 6. Save the retrieval manifest

* `data/thermal_data/image_coordinates.csv` — one row per training image, with
  its site assignment. This is what the dataset loader will use to find the
  right thermal window, and it serves the synthetic pipeline in
  `generated_data/` equally well.
* `data/thermal_data/themis_data/themis_query_sites.csv` — the deduplicated
  sites to actually query.

In [7]:
# map every image to the site that will hold its thermal data
site_lookup = query_sites.set_index(["site_lat", "site_lon"])["site_id"]
coordinates["site_id"] = coordinates.set_index(["site_lat", "site_lon"]).index.map(site_lookup)

coord_path = os.path.join(THERMAL_DIR, "image_coordinates.csv")
sites_path = os.path.join(THEMIS_DIR, "themis_query_sites.csv")

coordinates.to_csv(coord_path, index=False)
query_sites.to_csv(sites_path, index=False)

print("Saved:")
print(" ", coord_path, f"({len(coordinates)} rows)")
print(" ", sites_path, f"({len(query_sites)} rows)")

Saved:
  data\thermal_data\image_coordinates.csv (2374 rows)
  data\thermal_data\themis_data\themis_query_sites.csv (357 rows)


## 7. Phase 1 — find THEMIS observations per site

`themis_measurements.py` queries the ODE ArcGIS IRPBT4 footprint layer for one
point and returns the observations covering it, with acquisition time and Mars
local solar time. Local solar time is the useful axis here: the diagnostic
signal for a cave-connected pit is that it stays anomalously warm at night, so a
sequence spanning different times of day is what the temporal model needs.

**These are live network calls to an external archive.** The loop is disabled by
default -- set `RUN_PHASE_1 = True` to actually fetch, and consider starting with
`query_sites.head(...)` rather than the full set.

In [8]:
from themis_measurements import query_point, build_records

RUN_PHASE_1 = True          # set True to hit the network
SITES_TO_QUERY = query_sites  # e.g. query_sites.head(5) for a trial run

observations_path = os.path.join(THEMIS_DIR, "themis_observations.csv")

if RUN_PHASE_1:
    found = []

    for site in tqdm(
        SITES_TO_QUERY.itertuples(index=False),
        total=len(SITES_TO_QUERY),
        desc="Phase 1: footprint queries",
    ):
        try:
            payload = query_point(site.site_lat, site.site_lon)
            for record in build_records(payload, site.site_lon):
                record["site_id"] = site.site_id
                record["site_lat"] = site.site_lat
                record["site_lon"] = site.site_lon
                found.append(record)
        except Exception as exc:
            print(f"  {site.site_id} failed: {exc}")

    observations = pd.DataFrame(found)
    observations.to_csv(observations_path, index=False)
    print(f"\n{len(observations)} observations across "
          f"{observations['site_id'].nunique()} sites -> {observations_path}")
else:
    print("Phase 1 skipped (RUN_PHASE_1 is False).")
    if os.path.exists(observations_path):
        observations = pd.read_csv(observations_path)
        print(f"Loaded {len(observations)} previously saved observations.")
    else:
        observations = pd.DataFrame()
        print("No saved observations yet.")

Phase 1: footprint queries: 100%|██████████| 357/357 [04:53<00:00,  1.22it/s]


3339 observations across 356 sites -> data\thermal_data\themis_data\themis_observations.csv


### What local solar times are actually available?

The diagnostic signal for a cave-connected pit is that it stays anomalously warm
at night, so what the temporal branch needs is *contrast across time of day*.

Do not assume a tidy midday / morning / midnight triple is obtainable. Mars
Odyssey flies a near sun-synchronous orbit, so a given site is revisited near
only a couple of local solar times -- broadly a day pass and a night pass about
12 h apart -- with that local time drifting slowly over the mission rather than
sweeping the whole diurnal cycle. Check the real distribution before choosing a
sequence policy.

In [9]:
if len(observations):
    lmst = observations["mars_lmst_decimal_hours"].dropna()

    counts, _ = np.histogram(lmst, bins=24, range=(0, 24))
    print("Observations by Mars local solar time (all sites):")
    for hour, count in enumerate(counts):
        if count:
            print(f"  {hour:02d}:00  {'#' * min(count, 60)} {count}")

    per_site_span = (
        observations.groupby("site_id")["mars_lmst_decimal_hours"]
        .agg(lambda s: s.max() - s.min())
    )
    print(f"\nLocal-time span within a site: median {per_site_span.median():.1f} h, "
          f"max {per_site_span.max():.1f} h")
else:
    print("Run Phase 1 first.")

Observations by Mars local solar time (all sites):
  03:00  ######################## 24
  04:00  ############################################################ 67
  05:00  ############################################################ 158
  06:00  ############################################################ 1221
  07:00  ############################################################ 221
  08:00  ############################################################ 78
  10:00  ############## 14
  15:00  ############################################################ 67
  16:00  ############################################################ 85
  17:00  ############################### 31
  18:00  ############################################################ 1221
  19:00  ############################################################ 151
  20:00  # 1

Local-time span within a site: median 12.3 h, max 14.9 h


### Choosing a sequence per site

Rather than targeting fixed clock hours that may not exist, this picks the `T`
observations that are maximally separated on the 24 h circle (greedy
farthest-point selection). That extracts whatever diurnal contrast a site
genuinely offers -- a day/night pair when that is all there is, a wider spread
when more exists -- and degrades gracefully instead of silently returning three
near-identical frames.

Frames are then ordered by local solar time so the TCN sees a coherent
progression rather than an arbitrary permutation.

In [10]:
SEQUENCE_LENGTH = 3  # must match Hirise_Dataset(sequence_length=...)


def circular_gap(a, b, period=24.0):
    """Shortest distance between two local-solar-time values, in hours."""
    diff = abs(a - b) % period
    return min(diff, period - diff)


def spread_over_local_time(group, n):
    """Pick n rows whose local solar times are as far apart as possible."""
    usable = group.dropna(subset=["mars_lmst_decimal_hours"])
    if len(usable) <= n:
        return usable.sort_values("mars_lmst_decimal_hours")

    hours = usable["mars_lmst_decimal_hours"].tolist()

    # Seed with the earliest pass, then repeatedly add the observation
    # furthest (in time-of-day) from everything already selected.
    picked = [int(np.argmin(hours))]
    while len(picked) < n:
        best = max(
            (i for i in range(len(hours)) if i not in picked),
            key=lambda i: min(circular_gap(hours[i], hours[j]) for j in picked),
        )
        picked.append(best)

    return usable.iloc[sorted(picked, key=lambda i: hours[i])]


if len(observations):
    chosen = (
        observations
        .groupby("site_id", group_keys=False)
        .apply(spread_over_local_time, n=SEQUENCE_LENGTH)
    )

    counts = chosen.groupby("site_id").size()
    print(f"Sites with a full sequence of {SEQUENCE_LENGTH}: "
          f"{(counts == SEQUENCE_LENGTH).sum()} / {len(counts)}")
    print(f"Sites with fewer (not enough passes): {(counts < SEQUENCE_LENGTH).sum()}")

    gaps = chosen.groupby("site_id")["mars_lmst_decimal_hours"].agg(
        lambda s: max(s) - min(s)
    )
    print(f"\nSelected local-time spread: median {gaps.median():.1f} h, "
          f"min {gaps.min():.1f} h, max {gaps.max():.1f} h")

    # Sites whose passes all land at one local time carry no diurnal contrast:
    # their frames are near-duplicates and the temporal branch has nothing to
    # learn from them. Worth excluding or flagging before training.
    flat = gaps[gaps < 0.5]
    print(f"Sites with effectively no diurnal contrast (<0.5 h spread): {len(flat)}")
    if len(flat):
        print(f"  e.g. {', '.join(flat.index[:5])}")

    display(chosen.head(10))
else:
    chosen = pd.DataFrame()
    print("Run Phase 1 first.")

Sites with a full sequence of 3: 336 / 356
Sites with fewer (not enough passes): 20

Selected local-time spread: median 12.0 h, min 0.0 h, max 14.8 h
Sites with effectively no diurnal contrast (<0.5 h spread): 26
  e.g. site_0047, site_0048, site_0071, site_0072, site_0075


C:\Users\Utente\AppData\Local\Temp\ipykernel_153952\536390225.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(spread_over_local_time, n=SEQUENCE_LENGTH)


,observation_id,product_id_raw,utc_start,utc_end,mars_lmst_decimal_hours,mars_lmst_hhmm,solar_longitude_deg,emission_angle_deg,incidence_angle_deg,phase_angle_deg,product_version,product_lid,ode_id,site_id,site_lat,site_lon
8,I90212002,i90212002pbt,2022-04-16T08:15:57.830Z,2022-04-16T08:18:57.568Z,7.3011,07:18,209.778,1.587,71.050,71.498,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i902120...,55099005,site_0000,-79.75,234.95
19,I93121009,i93121009pbt,2022-12-11T20:51:41.273Z,2022-12-11T20:57:40.210Z,10.0888,10:05,352.644,1.727,84.634,84.562,2.0,urn:nasa:pds:ody.themis.geo:data_irpbt:i931210...,48882258,site_0000,-79.75,234.95
10,I91228004,i91228004pbt,2022-07-08T23:55:31.880Z,2022-07-08T23:57:31.880Z,17.2442,17:15,262.260,1.354,63.937,64.315,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i912280...,53569732,site_0000,-79.75,234.95
32,I90212002,i90212002pbt,2022-04-16T08:15:57.830Z,2022-04-16T08:18:57.568Z,7.3111,07:19,209.778,1.587,71.050,71.498,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i902120...,55099005,site_0001,-79.75,235.10
43,I93121009,i93121009pbt,2022-12-11T20:51:41.273Z,2022-12-11T20:57:40.210Z,10.0988,10:06,352.644,1.727,84.634,84.562,2.0,urn:nasa:pds:ody.themis.geo:data_irpbt:i931210...,48882258,site_0001,-79.75,235.10
34,I91228004,i91228004pbt,2022-07-08T23:55:31.880Z,2022-07-08T23:57:31.880Z,17.2542,17:15,262.260,1.354,63.937,64.315,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i912280...,53569732,site_0001,-79.75,235.10
56,I90212002,i90212002pbt,2022-04-16T08:15:57.830Z,2022-04-16T08:18:57.568Z,7.3144,07:19,209.778,1.587,71.050,71.498,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i902120...,55099005,site_0002,-79.75,235.15
67,I93121009,i93121009pbt,2022-12-11T20:51:41.273Z,2022-12-11T20:57:40.210Z,10.1021,10:06,352.644,1.727,84.634,84.562,2.0,urn:nasa:pds:ody.themis.geo:data_irpbt:i931210...,48882258,site_0002,-79.75,235.15
58,I91228004,i91228004pbt,2022-07-08T23:55:31.880Z,2022-07-08T23:57:31.880Z,17.2576,17:15,262.260,1.354,63.937,64.315,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i912280...,53569732,site_0002,-79.75,235.15
77,I89900002,i89900002pbt,2022-03-21T15:42:05.032Z,2022-03-21T15:45:04.770Z,7.2331,07:14,194.423,1.606,76.761,77.282,2.1,urn:nasa:pds:ody.themis.geo:data_irpbt:i899000...,55094825,site_0003,-79.70,234.95


## 8. Phase 2 — extract the thermal windows

`themis_id_to_thermal_value.py` downloads a whole PBT product and decodes it to
a `.npy`. That is far more than this pipeline needs: products run 0.4-126 MB
(median 22 MB) and the model uses a 32x32 patch, about 4 KB. The 312 products
behind a T=3 sequence cost roughly 7 GB of cache plus 7 GB of decoded arrays.

The ASU archive supports HTTP range requests, so GDAL can open a product through
`/vsicurl/` and fetch only the bytes covering the window we want.
`themis_windows.read_window_for_observation` does that -- about 2 s per window,
a few megabytes in total instead of ~14 GB, and nothing cached locally.

Values are **brightness temperature in Kelvin**, with **0 meaning no data**
(not "very cold"). Mask it before computing any statistic.

This is also the step that produces the final arrays, so the separate
window-cutting stage the earlier draft deferred is no longer needed.

In [ ]:
from themis_windows import read_window_for_observation, NODATA_KELVIN
from themis_id_to_thermal_value import ensure_index

RUN_PHASE_2 = True  # set True to fetch windows over the network
WINDOW_PX = 32

windows_dir = os.path.join(THEMIS_DIR, "windows")
os.makedirs(windows_dir, exist_ok=True)

if RUN_PHASE_2 and len(chosen):
    index_path = ensure_index()  # ~42 MB, cached after the first call
    manifest = []

    for site_id, group in tqdm(
        chosen.groupby("site_id"), desc="Phase 2: windows", total=chosen.site_id.nunique()
    ):
        frames, used = [], []

        for obs in group.itertuples():
            try:
                patch = read_window_for_observation(
                    obs.observation_id, obs.site_lat, obs.site_lon,
                    size=WINDOW_PX, index_path=index_path,
                )
            except Exception as exc:
                print(f"  {site_id}/{obs.observation_id}: {type(exc).__name__} {exc}")
                patch = None

            if patch is None:
                continue

            frames.append(patch)
            used.append({
                "observation_id": obs.observation_id,
                "mars_lmst_decimal_hours": obs.mars_lmst_decimal_hours,
                "valid_fraction": float((patch != NODATA_KELVIN).mean()),
                "kelvin_median": float(np.median(patch[patch != NODATA_KELVIN]))
                                 if (patch != NODATA_KELVIN).any() else None,
            })

        if not frames:
            continue

        # (T, 1, WINDOW_PX, WINDOW_PX) -- the shape the model's thermal branch takes
        stack = np.stack(frames)[:, None, :, :].astype(np.float32)
        np.save(os.path.join(windows_dir, f"{site_id}.npy"), stack)
        manifest.append({"site_id": site_id, "n_frames": len(frames), "frames": used})

    with open(os.path.join(THEMIS_DIR, "window_manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"\n{len(manifest)} sites written to {windows_dir}")
else:
    print("Phase 2 skipped (RUN_PHASE_2 is False, or no observations selected).")

Using cached ASU index: themis_phase2_cache\CMIDX_ODTIP.TAB


Phase 2: windows:   5%|▍         | 17/356 [00:22<05:55,  1.05s/it]

  site_0017/IA1672010PBT: ValueError Use a THEMIS observation ID such as I88015002


Phase 2: windows:   5%|▌         | 18/356 [00:28<14:21,  2.55s/it]

  site_0018/IA1672010PBT: ValueError Use a THEMIS observation ID such as I88015002


...

  site_0175/IA2159031PBT: ValueError Use a THEMIS observation ID such as I88015002


Phase 2: windows:  49%|████▉     | 176/356 [07:32<06:54,  2.30s/it]

  site_0176/IA1785019PBT: ValueError Use a THEMIS observation ID such as I88015002
  site_0176/IA2672001PBT: ValueError Use a THEMIS observation ID such as I88015002


Phase 2: windows:  51%|█████     | 180/356 [07:53<12:47,  4.36s/it]

  site_0179/IA2808023PBT: ValueError Use a THEMIS observation ID such as I88015002


## 9. Inspect what was extracted

Sanity-check the windows before training on them. What to look for:

* **Valid fraction** -- sites near a product edge lose pixels to no-data. A site
  whose windows are mostly zeros carries little signal.
* **Diurnal contrast** -- night frames should be markedly colder than day
  frames. Observed on real products: ~157-167 K at night against ~254-277 K in
  the afternoon. A site whose frames all sit at one temperature has no contrast
  for the temporal branch to use, whatever its local-time spread suggested.

In [ ]:
manifest_path = os.path.join(THEMIS_DIR, "window_manifest.json")

if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        manifest = json.load(f)

    frames = pd.DataFrame([
        {"site_id": entry["site_id"], **frame}
        for entry in manifest for frame in entry["frames"]
    ])

    print(f"{len(manifest)} sites, {len(frames)} frames")
    print(f"median valid fraction: {frames.valid_fraction.median():.2f}")
    print(f"sites with any frame <50% valid: "
          f"{frames[frames.valid_fraction < 0.5].site_id.nunique()}")

    span = frames.groupby("site_id")["kelvin_median"].agg(lambda s: s.max() - s.min())
    print(f"\nper-site temperature span (K): median {span.median():.1f}, "
          f"min {span.min():.1f}, max {span.max():.1f}")
    print(f"sites with <10 K span (little diurnal contrast): {(span < 10).sum()}")

    fig, axs = plt.subplots(1, 2, figsize=(14, 4))
    frames.kelvin_median.hist(bins=40, ax=axs[0])
    axs[0].set_title("Window median temperature")
    axs[0].set_xlabel("Kelvin")
    span.hist(bins=40, ax=axs[1])
    axs[1].set_title("Per-site day/night span")
    axs[1].set_xlabel("Kelvin")
    for ax in axs:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No windows yet -- run Phase 2 with RUN_PHASE_2 = True.")

## 10. Wiring the windows into training

`Hirise_Dataset` currently fabricates its thermal sequence
(`generate_synthetic_thermal=True`, a `torch.randn` placeholder shaped
`(T, 1, 32, 32)`). To train on the real data, load
`data/thermal_data/themis_data/windows/{site_id}.npy` instead, looking up each
image's `site_id` through `image_coordinates.csv`.

Two things to handle when doing that:

* **Sites with fewer than `T` frames.** Not every site yields a full sequence;
  the dataset returns a fixed `T`, so short sequences need padding, repetition,
  or exclusion. Padding with no-data zeros is the honest option, but keep it
  consistent with how the masked values are normalised.
* **Normalisation.** Mask `0` before any statistic. Prefer contrast within the
  window (pit against its surroundings) over absolute Kelvin: it is the
  physically invariant quantity, and it is robust to season, latitude, and the
  local-time drift across the mission.

The modality ablation in `model/training.ipynb` is the check that this actually
helped -- with the synthetic placeholder, "thermal" scores at chance and "both"
matches "optical". Real windows should move both.